In [1]:
import os
import math
import torch
from torch.utils.data import random_split, DataLoader
import numpy as np
import tempfile
from types import SimpleNamespace

from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from transformers.trainer_utils import RemoveColumnsCollator
from transformers.data.data_collator import default_data_collator

In [2]:
from tsfm_public.models.tspulse import TSPulseForClassification
from tsfm_public.toolkit.lr_finder import optimal_lr_finder

In [3]:
seed = 42
set_seed(seed)

In [4]:
from tsfm_public.toolkit.dataset import ClassificationDFDataset
from tsfm_public.toolkit.time_series_classification_preprocessor import TimeSeriesClassificationPreprocessor
from tsfm_public.toolkit.util import convert_tsfile_to_dataframe

In [5]:
dataset_name = "BasicMotions"

In [ ]:
path = f"/dccstor/tsfm23/datasets/UEA_Multivariate_ts/{dataset_name}/{dataset_name}_TRAIN.ts"

df_base = convert_tsfile_to_dataframe(     # returns the data correctly (in same order) from .ts file 
    path,
    return_separate_X_and_y=False,
)
label_column = "class_vals"
input_columns = [f"dim_{i}" for i in range(df_base.shape[1]-1)]

tsp = TimeSeriesClassificationPreprocessor(
    input_columns=input_columns,
    label_column=label_column,
    scaling=True,
)

tsp.train(df_base)     # working fine getting correct scaling factors
df_prep = tsp.preprocess(df_base)
base_dataset = ClassificationDFDataset(
    df_prep,
    id_columns=[],
    timestamp_column=None,
    input_columns=input_columns,
    label_column=label_column,
    context_length=512,
    static_categorical_columns=[],
    stride=1,
    enable_padding=False,
    full_series=True,
)

In [ ]:
import pandas as pd
print('Raw Dataframe : ')
df_first_elements = df_base.applymap(    lambda x: x if isinstance(x, str)
    else x.iloc[0] if isinstance(x, pd.Series) and not x.empty
    else None)
print(df_first_elements.head(10))
print('Dataframe after preprocessing : ')
df_first_elements = df_prep.applymap(    lambda x: x if isinstance(x, int)
    else x.iloc[0] if isinstance(x, pd.Series) and not x.empty
    else None)
print(df_first_elements.head(10))

Raw Dataframe : 
      dim_0     dim_1     dim_2     dim_3     dim_4     dim_5 class_vals
0  0.079106  0.394032  0.551444  0.351565  0.023970  0.633883   standing
1  0.377751 -0.610850 -0.147376 -0.103872 -0.109198 -0.037287   standing
2 -0.813905  0.825666  0.032712  0.021307  0.122515  0.775041   standing
3  0.289855  0.284130  0.213680 -0.314278  0.074574 -0.079901   standing
4 -0.123238  0.379341 -0.286006 -0.098545  0.058594 -0.074574   standing
5 -0.357300 -0.584885 -0.792751  0.074574  0.159802  0.023970   standing
6  1.236069 -0.569532  1.536733  0.143822  0.061258  0.905547   standing
7 -0.366403  0.331289 -0.817845 -0.167792 -0.093218  0.157139   standing
8 -0.342233  0.327415  0.157229  0.394179  0.074574 -0.037287   standing
9 -0.407421  1.413374  0.092782 -0.066584  0.223723  0.135832   standing
Dataframe after preprocessing : 
      dim_0     dim_1     dim_2     dim_3     dim_4     dim_5  class_vals  \
0 -0.349766  0.249919  0.444969  0.157446  0.026323  0.196120         

/tmp/ipykernel_1170646/210382264.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_first_elements = df_base.applymap(    lambda x: x if isinstance(x, str)
/tmp/ipykernel_1170646/210382264.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_first_elements = df_prep.applymap(    lambda x: x if isinstance(x, int)


In [ ]:
path = f"/dccstor/tsfm23/datasets/UEA_Multivariate_ts/{dataset_name}/{dataset_name}_TEST.ts"

df_test = convert_tsfile_to_dataframe(
    path,
    return_separate_X_and_y=False,
)
label_column = "class_vals"
input_columns = [f"dim_{i}" for i in range(df_test.shape[1]-1)]

tsp = TimeSeriesClassificationPreprocessor(
    input_columns=input_columns,
    label_column=label_column,
    scaling=True,
)

tsp.train(df_test)

df_prep = tsp.preprocess(df_test)

test_dataset = ClassificationDFDataset(
    df_prep,
    id_columns=[],
    timestamp_column=None,
    input_columns=input_columns,
    label_column=label_column,
    context_length=512,
    static_categorical_columns=[],
    stride=1,
    enable_padding=False,
    full_series=True,
)

In [ ]:
print(len(base_dataset))
print(len(test_dataset))

print('BASE DSET')
for i in range(len(base_dataset)):
    print(base_dataset[i]['past_values'][0], " : ", base_dataset[i]['target_values'].item())
print('TEST DSET')
for i in range(len(test_dataset)):
    print(test_dataset[i]['past_values'][0], " : ", test_dataset[i]['target_values'].item())

40
40
BASE DSET
tensor([-0.3498,  0.2499,  0.4450,  0.1574,  0.0263,  0.1961])  :  2
tensor([-0.3075,  0.1020,  0.2479, -0.0582, -0.0468,  0.0053])  :  2
tensor([-0.3185,  0.2990,  0.5373, -0.0481, -0.0176,  0.2060])  :  2
tensor([-0.0900,  0.1233,  0.1839,  0.0503, -0.0059, -0.0493])  :  2
tensor([-0.0468,  0.0766,  0.3296,  0.0175,  0.0527,  0.0802])  :  2
tensor([-0.1540,  0.4543,  0.0590,  0.0301,  0.1009,  0.0431])  :  2
tensor([-0.1932,  0.1939,  0.1829,  0.1650, -0.1258,  0.0484])  :  2
tensor([-0.4118,  0.2144,  0.2713,  0.0099, -0.0102,  0.0515])  :  2
tensor([-0.1672,  0.0367,  0.1622, -0.1540, -0.2194,  0.0992])  :  2
tensor([-0.3151,  0.0480, -0.0659, -0.3823,  0.1872,  0.1832])  :  2
tensor([-2.2638e-01, -1.3486e-04,  2.3022e-02, -1.9188e-01,  1.8284e-01,
         4.3888e-02])  :  1
tensor([-0.0063,  0.0822,  0.7263,  0.1713, -0.2267,  0.1840])  :  1
tensor([-0.4760,  0.3134,  0.2987,  0.0011,  0.0804,  0.2363])  :  1
tensor([-0.3711,  0.2442,  0.3670, -0.4983,  0.4213, -0

In [10]:
dataset_size = len(base_dataset)
print(dataset_size)
split_valid_ratio = 0.1
val_size = int(split_valid_ratio * dataset_size)  # 10% valid split
train_size = dataset_size - val_size
train_dataset, valid_dataset = random_split(base_dataset, [train_size, val_size])

40


In [11]:
print(len(train_dataset))
print(len(valid_dataset))
print(len(test_dataset))

print('TRAIN DSET')
for i in range(len(train_dataset)):
    print(train_dataset[i]['past_values'][0], " : ", train_dataset[i]['target_values'].item())
print('VAL DSET')
for i in range(len(valid_dataset)):
    print(valid_dataset[i]['past_values'][0], " : ", valid_dataset[i]['target_values'].item())
print('TEST DSET')
for i in range(len(test_dataset)):
    print(test_dataset[i]['past_values'][0], " : ", test_dataset[i]['target_values'].item())

36
4
40
TRAIN DSET
tensor([-0.3442,  0.2537,  0.2095,  0.0553, -0.0307,  0.0303])  :  3
tensor([-0.3151,  0.0480, -0.0659, -0.3823,  0.1872,  0.1832])  :  2
tensor([-0.4170,  0.0908,  0.1907, -0.0834, -0.0190,  0.0590])  :  0
tensor([-0.4596,  0.1092,  0.0215, -0.5576,  0.4125, -0.3757])  :  1
tensor([-0.3498,  0.2499,  0.4450,  0.1574,  0.0263,  0.1961])  :  2
tensor([-0.0900,  0.1233,  0.1839,  0.0503, -0.0059, -0.0493])  :  2
tensor([-0.1672,  0.0367,  0.1622, -0.1540, -0.2194,  0.0992])  :  2
tensor([-0.3871,  0.2006,  0.3600,  0.0641,  0.1053,  0.2241])  :  1
tensor([-0.2692,  0.0452,  0.1078, -0.0569,  0.0439,  0.0068])  :  1
tensor([-0.3711,  0.2442,  0.3670, -0.4983,  0.4213, -0.2189])  :  1
tensor([-0.4093,  0.2401,  0.3338,  0.1776,  0.0541,  0.0053])  :  0
tensor([-0.3185,  0.2990,  0.5373, -0.0481, -0.0176,  0.2060])  :  2
tensor([-0.4118,  0.2144,  0.2713,  0.0099, -0.0102,  0.0515])  :  2
tensor([-0.0468,  0.0766,  0.3296,  0.0175,  0.0527,  0.0802])  :  2
tensor([-0.3200